In [ ]:

import time
from pathlib import Path
import cv2
import mediapipe as mp
import pygame

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

UMBRAL_BOCA = 0.35


pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
class ControlesFaciales:
    def __init__(self):
        
        modelo = Path("face_landmarker.task")
        options = FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=str(modelo)),
            running_mode=VisionRunningMode.VIDEO,
            num_faces=1,
            output_face_blendshapes=True
        )

        self.cap = cv2.VideoCapture(0)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        self.cap.set(cv2.CAP_PROP_FPS, 30)
        self.ultimo_timestamp = -1

        try:
            if not self.cap.isOpened():
                raise RuntimeError("No se pudo abrir la camara")
            self.landmarker = FaceLandmarker.create_from_options(options)
        except BaseException:
            self.cap.release()
            raise

    def actualizar(self):
        # Valores seguros si no se detecta ningun rostro.
        datos = {"rostro": False, "posicion_x": 0.5,
                 "disparar": False, "escudo": False,
                 "boca": 0.0}

        ret, frame = self.cap.read()
        if not ret:
            return datos, None

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        timestamp = max(int(time.monotonic() * 1000),
                        self.ultimo_timestamp + 1)
        self.ultimo_timestamp = timestamp
        result = self.landmarker.detect_for_video(mp_image, timestamp)

        if result.face_landmarks:
            h, w = frame.shape[:2]
            for face in result.face_landmarks:
                for lm in face:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)),
                               1, (0, 255, 0), -1)

            nariz = result.face_landmarks[0][1]
            cv2.circle(frame, (int(nariz.x * w), int(nariz.y * h)),
                       6, (0, 0, 255), -1)

            movimientos = {}
            if result.face_blendshapes:
                movimientos = {b.category_name: b.score
                               for b in result.face_blendshapes[0]}

            boca = movimientos.get("jawOpen", 0)
            datos.update(rostro=True, posicion_x=nariz.x,
                         boca=boca,
                         disparar=boca > UMBRAL_BOCA)

        return datos, frame

    def cerrar(self):
        try:
            self.landmarker.close()
        finally:
            self.cap.release()

Videogame

In [ ]:
def jugar():
    pygame.init()

    # Configuracion original del juego y panel lateral para la webcam.
    ANCHO = 800
    ALTO = 600
    PANEL = 220
    FPS = 60
    pantalla = pygame.display.set_mode((ANCHO + PANEL, ALTO))
    pygame.display.set_caption("Face Invaders")
    reloj = pygame.time.Clock()
    fuente = pygame.font.Font(None, 32)
    fuente_panel = pygame.font.Font(None, 24)

    NEGRO = (10, 10, 25)
    BLANCO = (255, 255, 255)
    AZUL = (70, 150, 255)
    VERDE = (80, 220, 100)
    AMARILLO = (255, 220, 0)
    ROJO = (255, 100, 100)

    # Rango inicial de movimiento de la nariz en la webcam.
    X_MIN = 0.25
    X_MAX = 0.75
    nave = pygame.Rect(375, 540, 50, 30)
    posicion_nave = float(nave.centerx)

    disparos = []
    puntuacion = 0
    vidas = 3
    nivel = 1
    direccion = 1
    ultimo_disparo = 0
    ejecutando = True
    game_over = False


    def crear_enemigos():
        enemigos = []
        for fila in range(3):
            for columna in range(8):
                enemigo = pygame.Rect(80 + columna * 75,
                                      60 + fila * 50, 40, 25)
                enemigos.append(enemigo)
        return enemigos


    def texto_panel(texto, y, color=BLANCO):
        pantalla.blit(fuente_panel.render(texto, True, color), (ANCHO + 10, y))


    enemigos = crear_enemigos()
    controles = None

    try:
        controles = ControlesFaciales()

        while ejecutando:
            # Compensar los FPS de la camara (velocidades originales a 60 FPS).
            dt = min(reloj.tick(FPS) / 1000.0, 0.05)
            factor = dt * 60

            for evento in pygame.event.get():
                if evento.type == pygame.QUIT:
                    ejecutando = False

            teclas = pygame.key.get_pressed()
            if teclas[pygame.K_ESCAPE] or not ejecutando:
                break

            # NUEVO: obtener los controles del rostro en cada iteracion.
            datos, frame = controles.actualizar()
            pausa = not datos["rostro"]

            if not game_over and not pausa:
                # Convertir la coordenada de la nariz al rango de la pantalla.
                x = (datos["posicion_x"] - X_MIN) / (X_MAX - X_MIN)
                x = max(0.0, min(1.0, x))
                objetivo = nave.width / 2 + x * (ANCHO - nave.width)

                # Suavizar el movimiento para evitar saltos bruscos.
                posicion_nave += 0.25 * (objetivo - posicion_nave)
                nave.centerx = round(posicion_nave)
                nave.clamp_ip(pygame.Rect(0, 0, ANCHO, ALTO))

                tiempo_actual = pygame.time.get_ticks()
                if datos["disparar"] and tiempo_actual - ultimo_disparo >= 300:
                    disparo = pygame.Rect(nave.centerx - 3, nave.top, 6, 15)
                    disparos.append(disparo)
                    ultimo_disparo = tiempo_actual

                # Movimiento de los disparos 
                for disparo in disparos[:]:
                    disparo.y -= max(1, round(10 * factor))
                    if disparo.bottom < 0:
                        disparos.remove(disparo)

                velocidad_enemigos = 1 + nivel
                paso = max(1, round(velocidad_enemigos * factor)) * direccion
                bajar = any(e.right + paso > ANCHO or e.left + paso < 0
                            for e in enemigos)
                if bajar:
                    direccion *= -1
                    for enemigo in enemigos:
                        enemigo.y += 25
                else:
                    for enemigo in enemigos:
                        enemigo.x += paso

                for disparo in disparos[:]:
                    for enemigo in enemigos[:]:
                        if disparo.colliderect(enemigo):
                            enemigos.remove(enemigo)
                            disparos.remove(disparo)
                            puntuacion += 10
                            break

                if not enemigos:
                    nivel += 1
                    enemigos = crear_enemigos()
                    disparos.clear()
                    direccion = 1

                if any(e.bottom >= nave.top for e in enemigos):
                    vidas -= 1
                    enemigos = crear_enemigos()
                    disparos.clear()
                    direccion = 1
                    if vidas <= 0:
                        game_over = True

            pantalla.fill(NEGRO)

            if not game_over:
                pygame.draw.polygon(pantalla, AZUL, [
                    (nave.centerx, nave.top),
                    (nave.left, nave.bottom),
                    (nave.right, nave.bottom)])
                for enemigo in enemigos:
                    pygame.draw.rect(pantalla, VERDE, enemigo)
                for disparo in disparos:
                    pygame.draw.rect(pantalla, AMARILLO, disparo)

                if pausa:
                    texto = fuente.render("PAUSA - Rostro no detectado", True, BLANCO)
                    pantalla.blit(texto, texto.get_rect(center=(ANCHO // 2, ALTO // 2)))
            else:
                texto = fuente.render("GAME OVER", True, BLANCO)
                pantalla.blit(texto, (330, 260))
                texto = fuente.render("Pulsa R para reiniciar", True, BLANCO)
                pantalla.blit(texto, (280, 310))
                if teclas[pygame.K_r]:
                    puntuacion = 0
                    vidas = 3
                    nivel = 1
                    enemigos = crear_enemigos()
                    disparos.clear()
                    nave.x = 375
                    posicion_nave = float(nave.centerx)
                    direccion = 1
                    game_over = False

            texto = fuente.render(
                f"Puntos: {puntuacion}   Vidas: {vidas}   Nivel: {nivel}",
                True, BLANCO)
            pantalla.blit(texto, (10, 10))

            # NUEVO: mostrar la webcam en Pygame sin otra ventana de OpenCV.
            pygame.draw.rect(pantalla, (25, 30, 48), (ANCHO, 0, PANEL, ALTO))
            texto_panel("CONTROL FACIAL", 20)
            if frame is not None:
                imagen = cv2.resize(frame, (200, 150))
                imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
                superficie = pygame.surfarray.make_surface(imagen.swapaxes(0, 1))
                pantalla.blit(superficie, (ANCHO + 10, 55))
            else:
                texto_panel("Camara sin imagen", 100, ROJO)

            if datos["rostro"]:
                texto_panel("Rostro detectado", 225, VERDE)
                texto_panel(f"Nariz X: {datos['posicion_x']:.2f}", 260)
                texto_panel(f"Boca: {datos['boca']:.2f}", 290)
                if datos["disparar"]:
                    texto_panel("DISPARANDO", 360, AMARILLO)
            else:
                texto_panel("Rostro no detectado", 225, ROJO)

            texto_panel("Mueve la cabeza", 435)
            texto_panel("Abre la boca: disparo", 465)
            texto_panel("R: reiniciar al perder", 495)
            texto_panel("ESC: salir", 525)
            pygame.display.flip()

    except KeyboardInterrupt:
        print("Partida interrumpida")
    finally:
        try:
            if controles is not None:
                controles.cerrar()
        finally:
            pygame.quit()

In [5]:
jugar()